# Ising flow — Julia + Makie

In [ ]:
# Run once to install dependencies.
using Pkg
# Pkg.add(["NPZ", "Interpolations", "Meshing", "GeometryBasics", "LaTeXStrings"])

In [ ]:
using NPZ
using Interpolations
using Meshing
using GeometryBasics
using Random
using LaTeXStrings

# =================================================================
# PICK ONE BACKEND — uncomment exactly one block below.
# RESTART THE KERNEL before re-running after a change; Makie backends
# can't cleanly swap in a live session.
# Each block defines `show_fig(fig)` — figure cells call that.
# =================================================================

In [ ]:
# # --- A: GLMakie popup (interactive OS window) ---  [DEFAULT, verified]

using GLMakie
GLMakie.activate!()
Makie.inline!(false)
# show_fig(fig) = display(GLMakie.Screen(), fig)
show_fig(fig) = display(fig)

$$\frac{d\hat r}{d\ell}=2\hat r+3\tilde u-3\tilde u\hat r,\qquad
\frac{d\tilde u}{d\ell}=(4-d)\tilde u+10\tilde v-9\tilde u^2,\qquad
\frac{d\tilde v}{d\ell}=(6-2d)\tilde v-45\tilde u\tilde v+27\tilde u^3$$

In [ ]:
d = 3.5
ϵ = 4 - d

βr(r, u, v) = 2*r + 3*u - 3*u*r
βu(r, u, v) = (4-d)*u + 10*v - 9*u^2
βv(r, u, v) = (6-2*d)*v - 45*u*v + 27*u^3 
# βv(r, u, v) = 0

# Flow on the RG "upward" direction (negative beta).
V(r,u,v) = (βr(r,u,v), βu(r,u,v), βv(r,u,v))

In [ ]:
r0, u0, v0 = 0.5, 0.2, 0.2

βv(r0, u0, v0)

In [ ]:
function simulate(V, r0, u0, v0; dt=0.002, nsteps=1000)
    traj = zeros(nsteps, 3)
    traj[1, :] = [r0, u0, v0]
    for i in 2:nsteps
        r, u, v = traj[i-1, :]
        dr, du, dv = V(r, u, v)
        traj[i, :] = [r + dt*dr, u + dt*du, v + dt*dv]
    end
    return traj
end

function get_init(N, rr, ur, vr; rng=Random.default_rng())
    inits = Vector{NTuple{3,Float64}}()
    for _ in 1:N
        r0 = rand(rng) * (rr[2] - rr[1]) + rr[1]
        u0 = rand(rng) * ur
        v0 = rand(rng) * vr
        push!(inits, (r0, u0, v0))
    end
    return inits
end

In [ ]:
function plot_flow3D!(ax, N, trajs;
                       nsteps=1000, cmap=:plasma, crange=(-10.0, 10.0),
                       seed=nothing, linewidth=3, subsample=5, ms=10)

    cmin, cmax = Float32(crange[1]), Float32(crange[2])

    # Build one big NaN-separated strip so we get a single lines! draw call.
    # NOTE: NaN goes only in x/y/z (that's what breaks the line); the color array
    # uses a finite sentinel because NaN in per-vertex color can confuse shaders.
    segs = [traj[1:subsample:end, :] for traj in trajs]
    npts = sum(size(s, 1) for s in segs) + length(segs)  # +1 sep per traj

    xs = Vector{Float32}(undef, npts)
    ys = Vector{Float32}(undef, npts)
    zs = Vector{Float32}(undef, npts)
    cs = Vector{Float32}(undef, npts)

    i = 1
    for s in segs
        n = size(s, 1)
        @inbounds for k in 1:n
            xs[i] = s[k, 2]
            ys[i] = s[k, 3]
            zs[i] = s[k, 1]
            cs[i] = clamp(Float32(s[k, 1]), cmin, cmax)
            i += 1
        end
        # Strip separator: NaN in position only.
        xs[i] = NaN32; ys[i] = NaN32; zs[i] = NaN32; cs[i] = cmin
        i += 1
    end

    start_x = Float32[t[1, 2]     for t in trajs]
    start_y = Float32[t[1, 3]     for t in trajs]
    start_z = Float32[t[1, 1]     for t in trajs]
    end_x   = Float32[t[end, 2]   for t in trajs]
    end_y   = Float32[t[end, 3]   for t in trajs]
    end_z   = Float32[t[end, 1]   for t in trajs]
    end_c   = Float32[clamp(Float32(t[end, 1]), cmin, cmax) for t in trajs]

    lines!(ax, xs, ys, zs; color=cs, colormap=cmap, colorrange=crange, linewidth=linewidth,fxaa=true)
    scatter!(ax, start_x, start_y, start_z; color=:blue, markersize=ms)
    scatter!(ax, end_x, end_y, end_z;color=end_c, colormap=cmap, colorrange=crange, markersize=ms)

    return trajs
end

In [ ]:
# Flat arrow head for the streamplot: base spanning x ∈ [-½, ½] at z = 0, tip at
# z = 1 (the marker convention: +z is the arrow direction). Both windings are
# included so it stays visible from either side. Since the flow lies in the
# v = 0 plane the head ends up flat in that plane too, which reads much better
# than Makie's default 3D cone seen edge-on.
arrow_head_2d() = GeometryBasics.Mesh(
    Point3f[(-0.5, 0, 0), (0.5, 0, 0), (0, 0, 1)],
    GLTriangleFace[(1, 2, 3), (3, 2, 1)])

# Streamlines of (βu, βr) on the v = 0 plane, drawn into the 3D axis.
# streamplot! needs a 3D field on an Axis3, so the v-component is zeroed and the
# v-interval is degenerate — every streamline then stays exactly at v = 0.
# NOTE: v = 0 is not invariant (βv = 27u³ there); this is the projected (u, r)
# flow in that slice, not a sub-flow of the full system.
#
# Arrow knobs, all in data units (u/r units, *not* pixels):
#   arrow_width, arrow_length  — size of the head; start here when tuning.
#   arrow_head                 — swap in `Makie.automatic` for the default cone,
#                                or any mesh/marker.
#   gridsize, density          — how many streamlines, hence how many arrows.
# arrow_size=automatic is useless here: Makie scales it by the smallest box
# width, which is the degenerate v-interval, so the arrows come out invisible.
function plot_stream_v0!(ax, urange, rrange;
                         gridsize=18, density=1.0, stepsize=0.001, maxsteps=2000,
                         linecolor=:gray55, linewidth=2,
                         arrow_width=0.008, arrow_length=0.02,
                         arrow_head=arrow_head_2d())

    # streamplot passes points as (x, y, z) = (u, v, r), and wants the same back.
    f(p) = Point3f(βu(p[3], p[1], 0.0), 0.0, βr(p[3], p[1], 0.0))

    streamplot!(ax, f, urange[1]..urange[2], -1.0f-6..1.0f-6, rrange[1]..rrange[2];
                gridsize=(gridsize, 1, gridsize), density=density,
                stepsize=stepsize, maxsteps=maxsteps,
                color=p -> linecolor, linewidth=linewidth,
                arrow_head=arrow_head,
                arrow_size=Vec3f(arrow_width, arrow_width, arrow_length))
end

In [ ]:
yellow      = colorant"#fff800"
pink        = colorant"#ffdcfe"
pink        = colorant"#ffdcfe"
pink        = colorant"#f03ed4"
blue        = colorant"#a3f8ff"
green       = colorant"#c8ffce"
turquise    = colorant"#41e3c0"
red         = RGBf(0.8,0.2,0.2)

# cmap = cgrad([blue, pink, green])

In [ ]:
cmap = cgrad([blue, blue, pink, green, green], [0.0, 0.1, .5, .9, 1.0])
cmap = cgrad([:black, :black],[0, 1])

In [ ]:
set_theme!(theme_latexfonts())

rr = (-0.3, 0.05)
ur = 0.1
vr = 0.03

ms = 20
lw = 5

N = 100

function get_init(N, rr, ur, vr; rng=Random.default_rng())
    return [
        (-0.1, 0.001, 0.02),
        (-0.0, 0.001, 0.02),
        (-0.1, 0.001, 0.01),
        (-0.0, 0.001, 0.01),
        (-0.05, 0.001, 0.015), 
    ]
end

N = get_init(0,0,0,0);

nsteps = 600

rng = Random.MersenneTwister(1)
inits = get_init(N, rr, ur, vr; rng=rng)
trajs = [simulate(V, r0, u0, v0; nsteps=nsteps) for (r0, u0, v0) in inits];

In [ ]:
# a b c

fig = Figure(size=(150*s, 100*s); fontsize = 30)
ax  = Axis3(fig[1, 1]; 
    xlabel=L"u", ylabel=L"v", zlabel=L"r", viewmode = :fit,aspect = :equal,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs, 
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

dr = (rr[2] - rr[1]) / 2

# Streamlines of the (u, r) flow on the v = 0 plane, under the trajectories.
plot_stream_v0!(ax, (0, 1.2*ur), (rr[1] - dr, rr[2] + dr);
    gridsize     = 18,      # more streamlines -> more arrows
    linecolor    = :black,
    linewidth    = 2,
    arrow_width  = 0.004,   # data units
    arrow_length = 0.02,    # data units
    )

 # Plot the flow trajectories.
trajs = plot_flow3D!(ax, N, trajs; linewidth=lw, ms=ms, cmap=cmap)

xlims!(ax, (0, 1.2*ur))
ylims!(ax, (0, max(0.01,vr)))
zlims!(ax, (rr[1] - dr, rr[2] + dr))

show_fig(fig)